# Steered Molecular Dynamics and Jarzynski Analysis

This notebook shows how nonequilibrium pulling trajectories can be converted into free-energy estimates. The workflow links the SMD campaign engine to the Jarzynski estimators implemented in the analysis layer.

## Thermodynamic Background

For nonequilibrium work values $W_j$ measured across replicate pulling trajectories, the Jarzynski equality states

$$\delta G = -k_B T\ln\left(\frac{1}{N}\sum_{j=1}^{N} e^{-\beta W_j}\right), \quad \beta = \frac{1}{k_B T}. $$

When the work distribution is narrow and approximately Gaussian, the second-order cumulant approximation is often a useful diagnostic:

$$\delta G \approx \langle W \rangle - \frac{\beta}{2}\sigma_W^2.$$

In [ ]:
# Resolve project root and import SMD and Jarzynski modules.
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, SMDConfig
from src.simulate.smd import run_smd_campaign
from src.analyze.jarzynski import jarzynski_free_energy, evaluate_convergence

In [ ]:
equilibrated_state_path = DATA_DIR / 'trajectories' / 'equilibration' / 'npt_demo' / 'npt_final_state.xml'
system_xml_path = DATA_DIR / 'topologies' / 'example_system.xml'
pull_group_1 = [0, 1, 2]
pull_group_2 = [10, 11, 12]
smd_config = SMDConfig(
    spring_constant_kj_mol_nm2=1500.0,
    pulling_velocity_nm_per_ps=0.002,
    pull_distance_nm=0.5,
    n_replicates=8,
    save_interval_ps=1.0,
)
smd_output_dir = DATA_DIR / 'trajectories' / 'smd' / 'demo'

# Heavy calculation cell: run only when input files are available and you intend to launch the campaign.
# smd_results = run_smd_campaign(
#     equilibrated_state_path,
#     system_xml_path,
#     smd_config,
#     pull_group_1,
#     pull_group_2,
#     smd_output_dir,
# )

In [ ]:
# Compute the Jarzynski free-energy estimate and convergence diagnostic.
work_values = np.array([42.1, 43.8, 41.7, 44.0, 43.2, 42.6, 43.5, 42.9], dtype=float)
jarzynski_result = jarzynski_free_energy(work_values, temperature_k=310.0)
convergence_result = evaluate_convergence(work_values, temperature_k=310.0, n_subsets=5)
jarzynski_result, convergence_result

## Interpreting Nonequilibrium Estimates

The exact exponential-average estimator is sensitive to rare low-work trajectories, so convergence should always be checked explicitly. The cumulant estimate is not a replacement for the exact equality, but it is a useful sanity check when the work distribution is close to normal and dissipation is modest.